###  Load the datasets 

Load the maternity survery questions and the responses from the data files

In [58]:
import pandas as pd
import matplotlib.pyplot as plt

questions = pd.read_csv('../data/ChildlessnessQuestions.csv')
responses = pd.read_csv('../data/ChildlessnessResponses.csv')



In [59]:
# Standardise column names to improve consistency and usability

questions.columns = questions.columns.str.strip().str.lower().str.replace(' ', '_')
responses.columns = responses.columns.str.strip().str.lower().str.replace(' ', '_')

In [60]:
questions.shape
questions.head()

,question_code,full_question,construct_name
0,Q1,Women choose not to have a child because they ...,Financial
1,Q2,Women choose not to have a child because it is...,Financial
2,Q3,Women who choose not to have a child due to fi...,Financial
3,Q4,Women in high income jobs choose not to have a...,Financial
4,Q5,"Childless women, who are infertile, choose not...",Financial


In [61]:
responses.shape
responses.head()

,gender,age,employment_status,race/ethnicity,relationship_status,currently_have_children,q1,q2,q3,q4,...,q21,q22,q23,q24,q25,q26,q27,q28,q29,q30
0,Female,19,Student,African American,"Single, but not in a relationship",No,5,2,5.0,5,...,3.0,5.0,2.0,5.0,2.0,2.0,3.0,1.0,4.0,3.0
1,Female,27,Currently unemployed,African American,"Single, but not in a relationship",No,4,4,5.0,5,...,3.0,4.0,3.0,1.0,2.0,2.0,1.0,2.0,1.0,1.0
2,Female,27,Employee,Haitian-American,"Single, but not in a relationship",No,1,1,4.0,1,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,1.0
3,Female,45,Employee,Haitian-American,Married,Yes,2,2,4.0,4,...,1.0,1.0,3.0,3.0,1.0,1.0,2.0,2.0,1.0,1.0
4,Female,31,Employee,African American,Married,Yes,3,3,5.0,3,...,2.0,1.0,3.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0


### Drop any null values

In [62]:
responses.isnull().sum()
responses.dropna(inplace=True)


In [63]:
# Identify question columns (e.g., Q1, Q2, ... Q30)
q_cols = [c for c in responses.columns if c.upper().startswith("Q")]

len(q_cols), q_cols[:10]


(30, ['q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10'])

In [64]:
# Convert wide responses (Q1..Q30) into long format (one row per person per question)
responses_long = responses.melt(
    id_vars=[c for c in responses.columns if c not in q_cols],  # keep demographics
    value_vars=q_cols,                                          # the question answers
    var_name="question_code",
    value_name="response_value"
)

responses_long.shape, responses_long.head()


((3090, 8),
    gender  age     employment_status    race/ethnicity  \
 0  Female   19               Student  African American   
 1  Female   27  Currently unemployed  African American   
 2  Female   27              Employee  Haitian-American   
 3  Female   45              Employee  Haitian-American   
 4  Female   31              Employee  African American   
 
                  relationship_status currently_have_children question_code  \
 0  Single, but not in a relationship                      No            q1   
 1  Single, but not in a relationship                      No            q1   
 2  Single, but not in a relationship                      No            q1   
 3                            Married                     Yes            q1   
 4                            Married                     Yes            q1   
 
   response_value  
 0              5  
 1              4  
 2              1  
 3              2  
 4              3  )

In [65]:
questions["question_code"] = questions["question_code"].astype(str).str.strip().str.upper()
responses_long["question_code"] = responses_long["question_code"].astype(str).str.strip().str.upper()



In [66]:
merged_qr = responses_long.merge(
    questions[["question_code", "full_question", "construct_name"]],
    on="question_code",
    how="left"
)


In [67]:
# Confirm all response question codes successfully matched a full question (0 = perfect match)

merged_qr["full_question"].isna().sum()


np.int64(0)

In [ ]:
merged_qr.columns


Index(['gender', 'age', 'employment_status', 'race/ethnicity',
       'relationship_status', 'currently_have_children', 'question_code',
       'response_value', 'full_question', 'construct_name'],
      dtype='object')